# PharmaFlow Foundation — Session 2.1 Solutions
### pandas DataFrames — Load, Explore & Filter

These are the instructor solutions for all six exercise tiers. Every cell below has
been run for real against `data/raw/adverse_events_10k.csv`, and the outputs shown
are actual output, not illustrative text.

**Dataset:** `adverse_events_10k.csv` — 6,218 rows. It is what the Week 1 Lab
pipeline produces from a real pull of 10,000 openFDA adverse-event reports
(1,250 per quarter, July 2024 to June 2026): Session 1.3's `flatten_record()`
keeps only reports with exactly one suspect drug and a named first reaction,
so 6,218 of the 10,000 survive. It has one column Session 1.3's version did not
keep — `patient_age_unit` — and Exercise 4 is about why that column matters.

**pandas version:** these outputs come from pandas 3.0.5. Older pandas prints
`object` where you see `str` below; the numbers are the same.


## Before You Run Anything — Environment Check

Same setup pattern as every Week 1 session: this cell finds the repo
automatically, no matter which folder your notebook tool started in. It also
sets two display options so wide tables print on one line instead of being cut
into pieces.

Run it first. If it prints an error instead of "You're ready to go.", stop and
fix your setup before continuing.


In [1]:
# --- Setup check: always run this cell first ---
import os
import pandas as pd

def find_repo_root(marker_folder="data", start=None):
    """
    Different notebook tools start a notebook's kernel in different folders,
    so we search upward from wherever we happened to start until we find a
    folder that contains `data` -- that only exists at the repo root, so
    wherever we find it IS the repo root.
    """
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(current, marker_folder)):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError(
                f"Could not find a '{marker_folder}' folder anywhere above "
                f"{os.getcwd()}. Make sure this notebook is somewhere inside "
                f"the pharmaflow-ai repo."
            )
        current = parent


REPO_ROOT = find_repo_root()
DATA_FILE = os.path.join(REPO_ROOT, "data", "raw", "adverse_events_10k.csv")

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

print("pandas version:", pd.__version__)
if os.path.exists(DATA_FILE):
    print("Data file found: data/raw/adverse_events_10k.csv")
    print("You're ready to go.")
else:
    print("Missing file:", DATA_FILE)
    print("Fix: put adverse_events_10k.csv in data/raw/ at the top level of the repo.")


pandas version: 3.0.5
Data file found: data/raw/adverse_events_10k.csv
You're ready to go.


## Warm-Up — Six canteen orders (no FDA data yet)

Before touching 6,218 real rows, practise every move on a table small enough to
check by eye: six invented orders from a college canteen. There is one repeated
row, one blank quantity, one quantity typed as a word, and dates stored as text.


In [2]:
# WARM-UP: tiny invented data, no FDA records yet.
toy_orders = pd.DataFrame({
    "order_id":   ["A1", "A2", "A3", "A3", "A4", "A5"],
    "item":       ["Masala Dosa", "Samosa", "Chai", "Chai", "Vada Pav", "Samosa"],
    "qty":        ["2", "1", "3", "3", None, "two"],
    "order_date": ["20261005", "20261005", "20261006", "20261006", "20261006", "20261007"],
})

print("shape (rows, columns):", toy_orders.shape)
print()
print("missing values per column:")
print(toy_orders.isnull().sum())
print()
print("exact repeated rows:", toy_orders.duplicated().sum())

orders = toy_orders.drop_duplicates()
print("rows after drop_duplicates():", len(orders))

orders = orders.assign(
    qty_num=pd.to_numeric(orders["qty"], errors="coerce"),
    order_date=pd.to_datetime(orders["order_date"], format="%Y%m%d"),
)
print()
print(orders)
print()
print("qty_num missing after conversion:", orders["qty_num"].isnull().sum())
print("orders with 2 or more items:", (orders["qty_num"] >= 2).sum())


shape (rows, columns): (6, 4)

missing values per column:
order_id      0
item          0
qty           1
order_date    0
dtype: int64

exact repeated rows: 1
rows after drop_duplicates(): 5

  order_id         item  qty order_date  qty_num
0       A1  Masala Dosa    2 2026-10-05      2.0
1       A2       Samosa    1 2026-10-05      1.0
2       A3         Chai    3 2026-10-06      3.0
4       A4     Vada Pav  NaN 2026-10-06      NaN
5       A5       Samosa  two 2026-10-07      NaN

qty_num missing after conversion: 2
orders with 2 or more items: 2


## Exercise 1 · Beginner — "What exactly did we receive?"

The Data Manager wants an inventory of the new file before anyone builds on it:
how many rows and columns, what type pandas gave each column, and how much is
missing. Load it twice: once with defaults, once telling pandas which columns
are codes and identifiers rather than numbers.


In [3]:
# Exercise 1: load, inventory, and catch what default type-guessing does.
df_naive = pd.read_csv(DATA_FILE)

print("shape:", df_naive.shape)
print()
print("dtypes pandas guessed:")
print(df_naive.dtypes)
print()
print("values in 'serious':", df_naive["serious"].unique().tolist())
print("rows where serious == '1' (text):", (df_naive["serious"] == "1").sum())
print("rows where serious == 1 (number):", (df_naive["serious"] == 1).sum())


shape: (6218, 8)

dtypes pandas guessed:
safety_report_id      int64
drug_name               str
reaction                str
serious             float64
patient_age         float64
patient_age_unit    float64
country                 str
event_date            int64
dtype: object

values in 'serious': [2.0, 1.0, nan]
rows where serious == '1' (text): 0
rows where serious == 1 (number): 2917


In [4]:
# Reload, keeping identifiers and codes as text.
TEXT_COLUMNS = {
    "safety_report_id": str,
    "serious": str,
    "patient_age_unit": str,
    "event_date": str,
}
df = pd.read_csv(DATA_FILE, dtype=TEXT_COLUMNS)

df.info()
print()
print("missing values per column:")
print(df.isnull().sum())
print()
print("first 3 rows:")
print(df.head(3))


<class 'pandas.DataFrame'>
RangeIndex: 6218 entries, 0 to 6217
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   safety_report_id  6218 non-null   str    
 1   drug_name         6218 non-null   str    
 2   reaction          6218 non-null   str    
 3   serious           6205 non-null   str    
 4   patient_age       3202 non-null   float64
 5   patient_age_unit  3202 non-null   str    
 6   country           5815 non-null   str    
 7   event_date        6218 non-null   str    
dtypes: float64(1), str(7)
memory usage: 388.8 KB

missing values per column:
safety_report_id       0
drug_name              0
reaction               0
serious               13
patient_age         3016
patient_age_unit    3016
country              403
event_date             0
dtype: int64

first 3 rows:
  safety_report_id drug_name               reaction serious  patient_age patient_age_unit country event_date
0         24032351  

## Exercise 2 · Intermediate — The quarterly safety review

The Pharmacovigilance Lead runs a quarterly review of **serious** reports. For
2026 Q1 she needs: how many serious reports were received, the five drugs that
appear most often among them, and confirmation that the file covers the dates
it claims to. `event_date` is text, so convert it to a real date first.


In [5]:
# Exercise 2: fix the date type, check coverage, filter to serious reports in 2026 Q1.
df["event_date"] = pd.to_datetime(df["event_date"], format="%Y%m%d")

print("event_date dtype now:", df["event_date"].dtype)
print("earliest:", df["event_date"].min().date(), "| latest:", df["event_date"].max().date())
print("distinct drugs in the file:", df["drug_name"].nunique())
print()
print("rows per quarter:")
print(df["event_date"].dt.to_period("Q").value_counts().sort_index())

q1_2026 = (df["event_date"] >= "2026-01-01") & (df["event_date"] <= "2026-03-31")
serious = df["serious"] == "1"

serious_q1 = df[serious & q1_2026]
print()
print("serious reports received in 2026 Q1:", len(serious_q1))
print("rows in 2026 Q1 with seriousness not recorded:", (q1_2026 & df["serious"].isnull()).sum())

same_with_query = df.query('serious == "1" and "2026-01-01" <= event_date <= "2026-03-31"')
print(".query() gives the same count:", len(same_with_query))

print()
print("top 5 drugs among those serious reports:")
print(serious_q1["drug_name"].value_counts().head(5))
print()
print("first 3 rows, review columns only (.loc + .iloc):")
print(serious_q1.loc[:, ["safety_report_id", "drug_name", "reaction", "event_date"]].iloc[:3])


event_date dtype now: datetime64[us]
earliest: 2024-07-01 | latest: 2026-06-04
distinct drugs in the file: 1125

rows per quarter:
event_date
2024Q3    752
2024Q4    741
2025Q1    797
2025Q2    774
2025Q3    730
2025Q4    769
2026Q1    833
2026Q2    822
Freq: Q-DEC, Name: count, dtype: int64

serious reports received in 2026 Q1: 248
rows in 2026 Q1 with seriousness not recorded: 7
.query() gives the same count: 248

top 5 drugs among those serious reports:
drug_name
CABOMETYX       16
DUPIXENT        12
WEGOVY          11
PROGESTERONE     9
SKYRIZI          7
Name: count, dtype: int64

first 3 rows, review columns only (.loc + .iloc):
     safety_report_id                       drug_name                 reaction event_date
4563         10067080                        LUCENTIS          Endophthalmitis 2026-01-19
4565         26219464  SODIUM ZIRCONIUM CYCLOSILICATE  End stage renal disease 2026-01-01
4567         26219466                      LEUPROLIDE       Pituitary apoplexy 2026-01-

## Exercise 3 · Advanced — Are duplicates inflating the counts?

Before the top-10-drugs table goes to Medical Affairs, the Data Manager asks:
"Are any reports in here twice?" Check exact duplicates first, then rows that
match on everything except the report ID — and decide what to do with them
rather than deleting automatically.


In [6]:
# Exercise 3: duplicate investigation -- measure, inspect, flag (not delete).
print("repeated safety_report_id values:", df.duplicated(subset="safety_report_id").sum())
print("rows identical in every column:", df.duplicated().sum())

CONTENT = ["drug_name", "reaction", "serious", "patient_age",
           "patient_age_unit", "country", "event_date"]
lookalike = df.duplicated(subset=CONTENT, keep=False)
print()
print("rows that match another row on everything except the ID:", lookalike.sum())
print("rows drop_duplicates(subset=CONTENT) would delete:", df.duplicated(subset=CONTENT).sum())

lookalike_rows = df[lookalike]
print("how many of those rows have a patient age recorded:", lookalike_rows["patient_age"].notnull().sum())

df["possible_duplicate"] = lookalike & df["patient_age"].notnull()
print()
print("rows flagged possible_duplicate (look-alike AND age recorded):", df["possible_duplicate"].sum())
print(df.loc[df["possible_duplicate"], ["safety_report_id", "drug_name", "reaction",
                                        "patient_age", "country", "event_date"]]
        .sort_values(["drug_name", "event_date"]).head(6))

top10 = df["drug_name"].value_counts().head(10)
after_blind_drop = df.drop_duplicates(subset=CONTENT)["drug_name"].value_counts()
impact = pd.DataFrame({
    "all_rows": top10,
    "after_blind_drop": after_blind_drop.reindex(top10.index),
})
impact["reports_lost"] = impact["all_rows"] - impact["after_blind_drop"]
print()
print("today's top 10 drugs -- what a blind drop_duplicates would do to them:")
print(impact)


repeated safety_report_id values: 0
rows identical in every column: 0

rows that match another row on everything except the ID: 577
rows drop_duplicates(subset=CONTENT) would delete: 386
how many of those rows have a patient age recorded: 24

rows flagged possible_duplicate (look-alike AND age recorded): 24
     safety_report_id            drug_name                      reaction  patient_age country event_date
5565         26575684      ALLEGRA ALLERGY                      Pruritus         81.0      US 2026-04-03
5575         26575702      ALLEGRA ALLERGY                      Pruritus         81.0      US 2026-04-03
197          24051378       BOTOX COSMETIC                 Off label use         39.0      US 2024-07-04
225          24051419       BOTOX COSMETIC                 Off label use         39.0      US 2024-07-04
3614         25750772  BRENTUXIMAB VEDOTIN  Hypersensitivity pneumonitis         21.0      SA 2025-09-02
3618         25750778  BRENTUXIMAB VEDOTIN  Hypersensitivity 

## Exercise 4 · Corporate Scenario — "Our patients can't be 29,044 years old"

Medical Affairs wants the median patient age for a slide. `describe()` shows a
problem first. Find it, trace it to its cause, convert every age to years, and
report the median honestly — including how many reports it is based on.


In [7]:
# Exercise 4: find the bad ages, trace the cause, convert to years.
df["patient_age"] = pd.to_numeric(df["patient_age"], errors="coerce")
print("patient_age as reported:")
print(df["patient_age"].describe().round(1))

print()
print("ages above 120, with their unit code:")
print(df.loc[df["patient_age"] > 120, ["drug_name", "patient_age", "patient_age_unit"]])

print()
print("unit codes among known ages:")
print(df["patient_age_unit"].value_counts())


patient_age as reported:
count     3202.0
mean        94.2
std       1004.1
min          0.0
25%         39.2
50%         60.0
75%         72.0
max      29044.0
Name: patient_age, dtype: float64

ages above 120, with their unit code:
          drug_name  patient_age patient_age_unit
134        TEZSPIRE      29044.0              804
289        TAGRISSO      27208.0              804
337        TAGRISSO        839.0              802
409         SYNAGIS        292.0              804
433         FASENRA      17934.0              804
528        LYNPARZA      27510.0              804
548  BYDUREON BCISE      24111.0              804

unit codes among known ages:
patient_age_unit
801    3123
800      48
802      23
804       8
Name: count, dtype: int64


In [8]:
# openFDA unit codes (openFDA drug event field reference):
# 800 Decade, 801 Year, 802 Month, 803 Week, 804 Day, 805 Hour
UNIT_TO_YEARS = {
    "800": 10,
    "801": 1,
    "802": 1 / 12,
    "803": 7 / 365.25,
    "804": 1 / 365.25,
    "805": 1 / (24 * 365.25),
}
df["patient_age_years"] = df["patient_age"] * df["patient_age_unit"].map(UNIT_TO_YEARS)

print("converted ages (years):")
print(df["patient_age_years"].describe().round(1))
print()
print("rows whose age was NOT in years:", (df["patient_age_unit"].notnull() & (df["patient_age_unit"] != "801")).sum())
print("ages known before / after conversion:", df["patient_age"].notnull().sum(), "/", df["patient_age_years"].notnull().sum())

print()
print("the 29,044 row now reads:",
      round(df.loc[df["patient_age"] == 29044, "patient_age_years"].iloc[0], 1), "years")
print("mean age,   raw numbers:", round(df["patient_age"].mean(), 1),
      "| in real years:", round(df["patient_age_years"].mean(), 1))
print("median age, raw numbers:", df["patient_age"].median(),
      "| in real years:", round(df["patient_age_years"].median(), 1))
print("based on", df["patient_age_years"].notnull().sum(), "of", len(df), "reports")

wrong_fill = df["patient_age_years"].fillna(0).median()
print()
print("median if blanks were filled with 0 (do not do this):", wrong_fill)

with_age = df.dropna(subset=["patient_age_years"])
dupixent = with_age[with_age["drug_name"] == "DUPIXENT"]
print("DUPIXENT: median", round(dupixent["patient_age_years"].median(), 1),
      "years from", len(dupixent), "of", (df["drug_name"] == "DUPIXENT").sum(), "reports")

df["country_display"] = df["country"].fillna("Not reported")
print()
print("country for display (blanks labelled, not guessed):")
print(df["country_display"].value_counts().head(4))


converted ages (years):
count    3202.0
mean       55.5
std        22.1
min         0.0
25%        41.0
50%        61.0
75%        72.0
max       101.0
Name: patient_age_years, dtype: float64

rows whose age was NOT in years: 79
ages known before / after conversion: 3202 / 3202

the 29,044 row now reads: 79.5 years
mean age,   raw numbers: 94.2 | in real years: 55.5
median age, raw numbers: 60.0 | in real years: 61.0
based on 3202 of 6218 reports

median if blanks were filled with 0 (do not do this): 7.0
DUPIXENT: median 46.0 years from 442 of 621 reports

country for display (blanks labelled, not guessed):
country_display
US              4234
Not reported     403
GB               253
EU               229
Name: count, dtype: int64


## Exercise 5 · Interview Challenge — A missing-value report that never breaks

Write `missing_report(df)`. It returns one row per column with `missing_count`
and `missing_pct` (rounded to 1 decimal), sorted with the most-missing column
first (ties by column name). Rules: no `for` or `while` loops, it must not change
the DataFrame it is given, and it must work on a DataFrame with zero rows.


In [9]:
# Exercise 5: vectorised, side-effect free, safe on an empty DataFrame.
def missing_report(frame):
    """Missing values per column, most-missing first. Never modifies `frame`."""
    counts = frame.isnull().sum()
    total = len(frame)
    pct = (counts / total * 100).round(1) if total else counts * 0.0
    report = pd.DataFrame({"missing_count": counts, "missing_pct": pct})
    report = report.rename_axis("column").reset_index()
    report = report.sort_values(["missing_count", "column"], ascending=[False, True])
    return report.set_index("column")


raw = pd.read_csv(DATA_FILE, dtype=TEXT_COLUMNS)
before = raw.copy()

print(missing_report(raw))
print()
print("empty DataFrame:")
print(missing_report(raw.iloc[0:0]))
print()
print("input unchanged:", raw.equals(before))
print("toy orders:")
print(missing_report(toy_orders))


                  missing_count  missing_pct
column                                      
patient_age                3016         48.5
patient_age_unit           3016         48.5
country                     403          6.5
serious                      13          0.2
drug_name                     0          0.0
event_date                    0          0.0
reaction                      0          0.0
safety_report_id              0          0.0

empty DataFrame:
                  missing_count  missing_pct
column                                      
country                       0          0.0
drug_name                     0          0.0
event_date                    0          0.0
patient_age                   0          0.0
patient_age_unit              0          0.0
reaction                      0          0.0
safety_report_id              0          0.0
serious                       0          0.0

input unchanged: True
toy orders:
            missing_count  missing_pct
column  

---
### Grading notes

- **Warm-Up:** 1 repeated row (A3), 5 rows after `drop_duplicates()`, and 2 missing
  quantities after `to_numeric(errors="coerce")` — the blank one and `"two"`. A student
  who reports 1 missing has checked before converting.
- **Exercise 1:** the default load makes `serious` a float column (`1.0` / `2.0` / `NaN`),
  so `== "1"` silently finds **0** rows and `== 1` finds 2,917. That silent 0 is the
  teaching point; a student who never notices it has not inspected `dtypes`. The 13
  missing `serious` values are why pandas chose float: an integer column cannot hold a blank.
- **Exercise 2:** 248 serious reports in 2026 Q1, plus 7 reports in that quarter whose
  seriousness was not recorded (they are in neither the serious nor the non-serious
  count, and the answer should say so). Top drug CABOMETYX (16), then DUPIXENT (12).
  `.query()` must give the same 248. Rows per quarter run from 730 to 833, so every
  quarter has data. Note that the most-reported drug overall (DUPIXENT) is not the
  most-reported among serious reports in this quarter.
- **Exercise 3:** 0 repeated IDs and 0 fully identical rows. 577 rows match another row
  on everything except the ID; a blind `drop_duplicates(subset=CONTENT)` would delete
  386 of them. Only 24 of the 577 have a patient age recorded, and those 24 are flagged
  for review, not deleted. The impact table is the business point. A blind drop takes
  RYALTRIS from 79 reports to 29, pushing it from 5th to 30th place, and takes CABOMETYX
  from 170 to 139, into a tie with SKYRIZI. All 79 RYALTRIS reports were received on the
  same day (2025-02-01), all from the US, none with an age: a bulk submission. Whether
  they describe 79 patients or fewer cannot be decided from these columns, which is
  exactly why the right answer is "flag and ask the safety team," not "delete."
- **Exercise 4:** 79 rows were not in years (48 decades, 23 months, 8 days). The bad
  ages wreck the mean (94.2 raw vs 55.5 in years) but barely move the median (60.0 vs
  61.0), which is itself worth discussing: the median resists a few wild values, the
  mean does not. Filling blank ages with 0 drags the median down to 7.0, because 48.5%
  of ages are blank. Any answer that fills age with 0 or with the mean is wrong.
- **Exercise 5:** the empty-DataFrame case must print 0 / 0.0 for every column, with no
  `ZeroDivisionError` and no `NaN`. Check the code for a `for` loop and for any in-place
  change (`inplace=True`, assigning into `frame`).
